### Create the User-Item Matrix

In [3]:
import pandas as pd

# Assuming df contains 'student_id', 'recommended_courses', and 'gpa'
df = pd.read_csv('C:/Users/Farooq/OneDrive/Documents/Final year project/Chosen Research paper/ProjectFile/Data/mock_data_test.csv')

# Create the user-item matrix (students x courses)
user_item_matrix = df.pivot_table(index='student_id', columns='recommended_courses', values='gpa', aggfunc='mean')

# Fill missing values with 0 (no interaction)
user_item_matrix.fillna(0, inplace=True)

df.head()


,student_id,student_name,age,email,gpa,Difficulty_level,recommended_courses,College_Name,Course_Subject
0,1,Gillie Axtens,20,gaxtens0@slate.com,4.2,Intermediate,Front end Development,CIT,Data Science
1,2,Adrea Witherby,19,awitherby1@businessweek.com,7.7,Intermediate,Front end Development,Meenakshi,Data Science
2,3,Arleen Bellhanger,19,abellhanger2@accuweather.com,10.0,Intermediate,NaN,SRM,Cloud Computing
3,4,Annetta Stanyland,20,astanyland3@shop-pro.jp,8.5,Beginner,SQL,SIET,Data Science
4,5,Ferrel Atack,19,fatack4@newyorker.com,4.5,Beginner,Big Data Analytics,Crescent,Data Science


### Similarity between Courses

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity between courses
course_similarity = cosine_similarity(user_item_matrix.T)  # Transpose to compare courses
course_similarity_df = pd.DataFrame(course_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

# Display the similarity matrix (courses x courses)
course_similarity_df


recommended_courses,AWS,Backend Development,Big Data Analytics,Deep Learning,Front end Development,Machine Learning,SQL
recommended_courses,,,,,,,
AWS,1.0,0.0,0.0,0.0,0.0,0.0,0.0
Backend Development,0.0,1.0,0.0,0.0,0.0,0.0,0.0
Big Data Analytics,0.0,0.0,1.0,0.0,0.0,0.0,0.0
Deep Learning,0.0,0.0,0.0,1.0,0.0,0.0,0.0
Front end Development,0.0,0.0,0.0,0.0,1.0,0.0,0.0
Machine Learning,0.0,0.0,0.0,0.0,0.0,1.0,0.0
SQL,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## Generating based on Similarity

In [5]:
# Function to recommend courses for a student
def recommend_courses(student_id, user_item_matrix, course_similarity_df, top_n=5):
    # Get the student's courses and their interaction (GPA)
    student_courses = user_item_matrix.loc[student_id]
    
    # Get the courses the student has interacted with (GPA > 0)
    interacted_courses = student_courses[student_courses > 0].index
    
    # Calculate the total similarity for each course not interacted with
    course_scores = {}
    for course in user_item_matrix.columns:
        if course not in interacted_courses:
            # Sum up the similarities for the courses the student has interacted with
            similarity_score = sum(course_similarity_df[course][interacted_courses] * student_courses[interacted_courses])
            course_scores[course] = similarity_score
    
    # Sort courses based on the similarity score and get the top N recommendations
    recommended_courses = sorted(course_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    return [course for course, score in recommended_courses]

# Example usage
student_id = 102
recommended_courses = recommend_courses(student_id, user_item_matrix, course_similarity_df)
print(f"Top 5 Recommended Courses for Student ID {student_id}: {recommended_courses}")


Top 5 Recommended Courses for Student ID 102: ['AWS', 'Big Data Analytics', 'Deep Learning', 'Front end Development', 'Machine Learning']
